Используем selenium для парсинга сайта hirify

In [ ]:
!pip install selenium

In [138]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import logging
import sys
import pandas as pd

https://discourse.jupyter.org/t/jupyter-does-not-print-logging-information/11790

In [192]:
BASE_URL = "https://hirify.me"
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger('selenium_scrapper')
logger.setLevel(logging.DEBUG)
def prepare_driver_page(period: str = "period=week", page: str = "1") -> webdriver.Chrome:
    driver = webdriver.Chrome()
    logger.info("Driver created")
    if page:
        url = f"{BASE_URL}?page={page}&{period}"
        period = f"page={page}&{period[1:]}"
    driver.get(url)
    time.sleep(2)
    logger.info("Page loaded")
    settings_button = driver.find_element(By.CLASS_NAME, "settings-button")
    settings_button.click()
    filter_up = driver.find_element(By.CLASS_NAME, "filters-row")
    filters = filter_up.find_elements(By.TAG_NAME, "input")
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
        if not f.is_selected():
            f.click()
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
    logger.info("Filters applied")
    return driver



In [140]:
def parse_page(driver: webdriver.Chrome):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.vacancy-card[data-vacancy-id]")
    logger.info(f"Found {len(cards)} cards")
    return cards

cards = parse_page(driver)

2026-04-07 02:50:57,775 Found 15 cards


Переписываем все это на 3 функции: 1 итератор по стринице 2 парсер 1 карточки 3 безопасные getter 

In [ ]:
def safe_find_element(parent, by, value):
    try:
        return parent.find_element(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements(parent, by, value):
    try:
        return parent.find_elements(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements_text(parent, by, value):
    r = safe_find_elements(parent, by, value)
    if r:
        if isinstance(r, list):
            return [elem.text for elem in r]
        else:
            return r.text
    return None

def parse_card(card):
    logger.debug(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
    vacancy_link = safe_find_element(card, By.CLASS_NAME, "vacancy-card-link")
    vacancy_title = safe_find_element(card, By.CLASS_NAME, "title")
    vacancy_tags_div = safe_find_element(card, By.CLASS_NAME, "common-tags")
    vacancy_tags = []
    if vacancy_tags_div:
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
    vacancy_company = safe_find_element(card, By.CLASS_NAME, "company").text
    vacancy_time = safe_find_element(card, By.CLASS_NAME, "date")
    vacancy_skills_div = safe_find_element(card, By.CLASS_NAME, "vacancy-tags")
    vacancy_skills = []
    if vacancy_skills_div:
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
    vacansy_salary = safe_find_elements_text(card, By.CLASS_NAME, "salary")
    return {
        "title": vacancy_title.text,
        "link": vacancy_link.get_attribute("href"),
        "tags": vacancy_tags,
        "company": vacancy_company,
        "time": vacancy_time.text,
        "skills": vacancy_skills,
        "salary": vacansy_salary
    }

def parse_vacancies(cards: list) -> list[dict]:
    vacancies = []
    for card in cards:
        parse_card(card)
        vacancies.append(parse_card(card))
    return vacancies

vacancies = parse_vacancies(cards)
df = pd.DataFrame(vacancies)
df.head(20)

Итак погинация

In [151]:
next_page_button = safe_find_element(driver, By.CSS_SELECTOR, 'button[aria-label="Next Page"]')
#next_page_button.click()
next_page_button.get_attribute("aria-label")

'Next Page'

листаем до последней страницы и находим по devtools что появляется флаг disabled

In [146]:
next_page_button.get_attribute("disabled")

In [ ]:
def safe_find_element(parent, by, value):
    try:
        return parent.find_element(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements(parent, by, value):
    try:
        return parent.find_elements(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements_text(parent, by, value):
    r = safe_find_elements(parent, by, value)
    if r:
        if isinstance(r, list):
            return [elem.text for elem in r]
        else:
            return r.text
    return None

def parse_card(card):
    logger.debug(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
    vacancy_link = safe_find_element(card, By.CLASS_NAME, "vacancy-card-link")
    vacancy_title = safe_find_elements_text(card, By.CLASS_NAME, "title")
    vacancy_tags_div = safe_find_element(card, By.CLASS_NAME, "common-tags")
    vacancy_tags = []
    if vacancy_tags_div:
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
    vacancy_company = safe_find_elements_text(card, By.CLASS_NAME, "company")
    vacancy_time = safe_find_element(card, By.CLASS_NAME, "date")
    vacancy_skills_div = safe_find_element(card, By.CLASS_NAME, "vacancy-tags")
    vacancy_skills = []
    if vacancy_skills_div:
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
    vacansy_salary = safe_find_elements_text(card, By.CLASS_NAME, "salary")
    return {
        "title": vacancy_title,
        "link": vacancy_link.get_attribute("href"),
        "tags": vacancy_tags,
        "company": vacancy_company,
        "time": vacancy_time.text,
        "skills": vacancy_skills,
        "salary": vacansy_salary
    }

def parse_vacancies(cards: list) -> list[dict]:
    vacancies = []
    for card in cards:
        parse_card(card)
        vacancies.append(parse_card(card))
    return vacancies

def parse_page(driver: webdriver.Chrome):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.vacancy-card[data-vacancy-id]")
    logger.info(f"Found {len(cards)} cards")
    return cards

def go_to_next_page(driver):
    next_page_button = safe_find_element(driver, By.CSS_SELECTOR, 'button[aria-label="Next Page"]')
    if next_page_button and not next_page_button.get_attribute("disabled"):
        next_page_button.click()
        return True
    return False

In [193]:
def parse_pages(driver):
    all_vacancies = []
    for i in range(1000):
        try:
            time.sleep(1)
            logger.info(f"Parsing page {i+1}")
            cards = parse_page(driver)
            vacancies = parse_vacancies(cards)
            all_vacancies.extend(vacancies)
            if not go_to_next_page(driver):
                break 
        except Exception as e:
            logger.error(f"Error parsing page {i+1}: {e}")
            break
    return all_vacancies

logger.setLevel(logging.DEBUG)
driver = prepare_driver_page(page=18)
vacancies = parse_pages(driver)
df = pd.DataFrame(vacancies)
df.to_csv('vacancies.csv')

2026-04-07 03:42:07,766 Driver created
2026-04-07 03:42:14,661 Page loaded
2026-04-07 03:42:14,761 Filter  is selected: True
2026-04-07 03:42:14,771 Filter  is selected: False
2026-04-07 03:42:14,819 Filter  is selected: True
2026-04-07 03:42:14,829 Filter  is selected: False
2026-04-07 03:42:14,875 Filter  is selected: True
2026-04-07 03:42:14,881 Filter  is selected: True
2026-04-07 03:42:14,888 Filter  is selected: True
2026-04-07 03:42:14,894 Filter  is selected: True
2026-04-07 03:42:14,895 Filters applied
2026-04-07 03:42:15,897 Parsing page 1
2026-04-07 03:42:15,908 Found 15 cards
2026-04-07 03:42:15,914 Parsing card with id 440575
2026-04-07 03:42:16,029 Parsing card with id 440575
2026-04-07 03:42:16,105 Parsing card with id 440574
2026-04-07 03:42:16,188 Parsing card with id 440574
2026-04-07 03:42:16,270 Parsing card with id 440573
2026-04-07 03:42:16,361 Parsing card with id 440573
2026-04-07 03:42:16,448 Parsing card with id 440572
2026-04-07 03:42:16,529 Parsing card with

из за [этой](https://hirify.me/jobs/440275-graficeskii-dizainer) идиотской белоруской ваки все падает надо в бизнес вывод не рекомендовать релов в минск 